<a href="https://colab.research.google.com/github/eeswepe/PCVK_GANJIL_2026/blob/main/Week02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Jobsheet Minggu 2 — Representasi dan Karakteristik Citra Digital

**Pengolahan Citra dan Visi Komputer — Jurusan Teknologi Informasi, Politeknik Negeri Malang**

Notebook ini mengikuti Modul 2 (D1). Jalankan sel dari atas ke bawah.

**Tujuan praktikum**
1. Menjelaskan representasi citra digital sebagai array numerik (biner, keabuan, berwarna).
2. Menjelaskan digitalisasi citra (sampling & kuantisasi) dan menghitung ukuran data mentah.
3. Menerapkan operasi dasar: konversi ruang warna, resize, flipping, dan penggambaran geometri/teks.

## D1.1 — Menghubungkan Colab dengan Google Drive

**Cara 1:** mount Google Drive sehingga berkas di dalamnya bisa diakses langsung.

In [108]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Cara 2 (alternatif):** unggah berkas langsung dari komputer lokal.

In [ ]:
from google.colab import files
uploaded = files.upload()   # pilih ktp.jpg / foto aktivitas Anda

## D1.2 — Menampilkan Gambar

Import pustaka yang dipakai sepanjang praktikum. Catatan penting:
- `cv2.imread()` menyimpan citra dalam urutan kanal **BGR**.
- `cv2_imshow()` (dari `google.colab.patches`) juga mengasumsikan **BGR**.
- `plt.imshow()` (Matplotlib) mengasumsikan **RGB**.

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
from skimage import io

# Ganti path ini dengan lokasi berkas Anda (copy path dari panel Files di Colab).
PATH = 'ktm.jpeg'

img = cv.imread(PATH)
cv2_imshow(img)                     # tampilan langsung (BGR)

print('Resolusi gambar: ', img.shape[1], 'x', img.shape[0])
print('Shape (tinggi, lebar, kanal):', img.shape)
print('dtype:', img.dtype)

Bila gambar punya path berbeda, ubah variabel `PATH`. Jika dibaca dengan
`skimage.io.imread()` hasilnya **sudah RGB**, jadi tidak perlu konversi.

In [ ]:
plt.imshow(img)     # format warna BGR -> R dan B terlihat tertukar
plt.title('Tanpa konversi (BGR dibaca sebagai RGB)')
plt.show()

img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
plt.imshow(img_rgb)
plt.title('Setelah konversi BGR -> RGB')
plt.show()

## D1.3 — Menampilkan Citra Grayscale

Array 2 dimensi tidak menyimpan informasi warna; warna tampilan ditentukan
oleh **colormap** yang dipilih saat menampilkan.

In [ ]:
img_gray = cv.imread(PATH, cv.IMREAD_GRAYSCALE)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(img_gray)                 # colormap default 'viridis'
plt.title('Tanpa cmap')

plt.subplot(1, 2, 2)
plt.imshow(img_gray, cmap='gray')    # hitam-putih sesuai mestinya
plt.title('Dengan cmap gray')

plt.show()

plt.imshow(img_gray, cmap='magma')   # pseudocolor
plt.title('Pseudocolor: magma')
plt.show()

## D1.4 — Resize

`cv.resize()` menerima ukuran dalam urutan **(lebar, tinggi)** — kebalikan dari `img.shape`.

In [ ]:
h, w = img.shape[:2]
img_output = cv.resize(img_rgb, (w // 2, h // 2))
plt.imshow(img_output)
plt.title('Setelah resize')
plt.show()

print('shape sebelum:', img_rgb.shape, '| sesudah:', img_output.shape)

## D1.5 — Flipping

`cv.flip(img, flipCode)`: `0` = vertikal (atas–bawah), `1` = horizontal (kiri–kanan), `-1` = keduanya.

In [ ]:
img_flip = cv.flip(img_output, 1)   # cermin horizontal
plt.imshow(img_flip)
plt.show()

fig = plt.figure(figsize=(10, 10))  # memperbesar kanvas tampilan saja
ax = fig.add_subplot(1, 1, 1)
ax.imshow(img_flip)
plt.show()

## D1.6 — Menggambar Bentuk Geometri dan Teks

Mulai dengan kanvas hitam. `cv2.rectangle()` dan kawan-kawan **menulis langsung**
ke array masukan, jadi gunakan `img.copy()` bila citra asli masih dibutuhkan.
Koordinat selalu dalam urutan **(x, y)**.

In [ ]:
black_img = np.zeros(shape=(530, 530, 3), dtype=np.uint8)
plt.imshow(black_img)
plt.show()

# Persegi panjang 1: pt1=(290,0) pt2=(520,130), warna (0,0,255) = MERAH pada kanal BGR
cv.rectangle(black_img, pt1=(290, 0), pt2=(520, 130), color=(0, 0, 255), thickness=10)

# Persegi panjang 2: pt1=(250,250) pt2=(350,350), warna (255,0,0) = BIRU pada kanal BGR
cv.rectangle(black_img, pt1=(250, 250), pt2=(350, 350), color=(255, 0, 0), thickness=15)
plt.imshow(black_img)
plt.show()

In [ ]:
# Lingkaran tepi: center=(150,150) radius=60 warna hijau
cv.circle(black_img, center=(150, 150), radius=60, color=(0, 255, 0), thickness=8)

# Lingkaran penuh (thickness=-1) dan garis diagonal
cv.circle(black_img, center=(400, 400), radius=50, color=(0, 255, 0), thickness=-1)
cv.line(black_img, pt1=(0, 0), pt2=(530, 530), color=(255, 255, 255), thickness=5)
plt.imshow(black_img)
plt.show()

In [ ]:
font = cv.FONT_HERSHEY_SIMPLEX
cv.putText(black_img, text='PCVK is fun!', org=(10, 500), fontFace=font, fontScale=2,
           color=(255, 255, 0), thickness=3, lineType=cv.LINE_AA)
plt.imshow(black_img)
plt.show()

### Polyline

`cv.polylines()` memerlukan array titik bertipe `int32` dengan bentuk (N, 1, 2),
dibungkus dalam sebuah list.

In [ ]:
black_img2 = np.zeros(shape=(530, 530, 3), dtype=np.int32)
plt.imshow(black_img2)
plt.show()

vertices = np.array([[150, 350], [250, 250], [450, 350], [250, 450]], dtype=np.int32)
pts = vertices.reshape((-1, 1, 2))
print(pts)

cv.polylines(black_img2, [pts], isClosed=True, color=(0, 255, 0), thickness=5)
plt.imshow(black_img2)
plt.show()

**Langkah 7.** Simpan notebook ini dengan nama `Week02.ipynb` melalui
*File → Save a copy in GitHub*.

---
# Pertanyaan Praktikum

Jawaban disertai keluaran program pada sel di bawah tiap pertanyaan.

### 1. Tampilkan satu citra bebas dengan `cv2_imshow()` dan `plt.imshow()` tanpa konversi warna apa pun. Mengapa hasilnya berbeda?

Keduanya membaca array yang sama, tetapi mengasumsikan urutan kanal berbeda:
`cv2_imshow()` menganggap kanal ke-0 = **B**, ke-2 = **R**; `plt.imshow()` menganggap
kanal ke-0 = **R**, ke-2 = **B**. Karena `cv.imread()` menyimpan BGR, kanal merah dan
biru tertukar pada tampilan Matplotlib — wajah/objek berwarna merah jadi biru.
Nilai piksel di memori **identik**; yang berbeda hanya tafsir saat menampilkan.

In [ ]:
cv2_imshow(img)                                  # tampil benar (BGR)
plt.imshow(img); plt.title('plt.imshow tanpa konversi'); plt.show()  # R/B tertukar

y, x = 100, 100
b, g, r = img[y, x]
print(f'Pixel (baris={y}, kolom={x}) dari cv.imread  -> B={b}, G={g}, R={r}')
print('Array yang sama juga dipakai cv2_imshow, jadi nilai B di kanal 0.')
print('plt.imshow membaca kanal 0 sebagai R, sehingga R dan B tampak tertukar.')

### 2. Buat kanvas hitam berukuran sama dengan dtype `int16` dan `int32`. Bandingkan `dtype`, `min()`, `max()`, dan `nbytes`. Apakah tampilannya berbeda? Apakah ukuran memorinya berbeda? Jelaskan.

- Tampilan **sama-sama hitam** (semua nilai 0).
- `int16` memakai **2 byte/piksel**, `int32` memakai **4 byte/piksel** → `nbytes` int32 = 2× int16.
- `min()`/`max()` keduanya 0 karena masih kosong (belum digambar).
- Alasan: `dtype` menentukan jangkauan nilai yang bisa disimpan sekaligus besar memori per elemen.

In [ ]:
canvas16 = np.zeros(shape=(530, 530, 3), dtype=np.int16)
canvas32 = np.zeros(shape=(530, 530, 3), dtype=np.int32)

for name, c in [('int16', canvas16), ('int32', canvas32)]:
    print(f'{name}: dtype={c.dtype}, min={c.min()}, max={c.max()}, nbytes={c.nbytes:,} byte')

print('Rasio memori int32/int16 =', canvas32.nbytes / canvas16.nbytes)
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1); plt.imshow(canvas16); plt.title('int16')
plt.subplot(1, 2, 2); plt.imshow(canvas32); plt.title('int32')
plt.show()

### 3. Apa kegunaan `from google.colab.patches import cv2_imshow`, dan mengapa `cv2.imshow()` bawaan OpenCV tidak dapat dipakai di Google Colab?

- `cv2_imshow()` adalah pengganti `cv2.imshow()` yang menampilkan citra sebagai **keluaran sel** notebook.
- `cv2.imshow()` membutuhkan **jendela GUI** (HighGUI) pada sistem operasi. Colab berjalan di server tanpa layar/X11,
  sehingga pemanggilannya gagal atau tidak menampilkan apa pun.
- Perlu diingat `cv2_imshow()` mengasumsikan masukan **BGR**, sedangkan `plt.imshow()` mengasumsikan **RGB**.

In [ ]:
# cv.imshow('w', img)  # -> error / tidak tampil di Colab (tidak ada GUI)
cv2_imshow(img)        # pengganti yang bekerja di Colab

### 4. Citra yang dibaca dengan `skimage.io.imread()` ditampilkan berdampingan dengan hasil `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` atas citra yang sama. Manakah yang menampilkan warna sebenarnya, dan manakah yang kanalnya tertukar? Buktikan dengan nilai piksel, bukan pengamatan visual.

- `skimage.io.imread()` sudah **RGB** → menampilkan warna sebenarnya.
- `cv.cvtColor(img_bgr, COLOR_BGR2RGB)` juga menghasilkan **RGB**, jadi keduanya benar.
- Yang **tertukar** adalah `cv.imread()` mentah (BGR) yang ditampilkan oleh `plt.imshow()`
  (tanpa konversi). Bukti: bandingkan nilai kanal pada piksel yang sama.

In [ ]:
img_skimage = io.imread(PATH)                  # RGB
img_bgr = cv.imread(PATH)                      # BGR
img_bgr2rgb = cv.cvtColor(img_bgr, cv.COLOR_BGR2RGB)

y, x = 100, 100
print('skimage  (R,G,B) =', img_skimage[y, x][:3])
print('cv2 BGR mentah   =', img_bgr[y, x])
print('cv2 BGR->RGB     =', img_bgr2rgb[y, x])
print('Selisih maks skimage vs BGR->RGB =', np.abs(img_skimage[..., :3].astype(int) - img_bgr2rgb.astype(int)).max())

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(img_skimage); plt.title('skimage (RGB benar)')
plt.subplot(1, 2, 2); plt.imshow(img_bgr2rgb); plt.title('cv2 BGR->RGB (RGB benar)')
plt.show()

plt.imshow(img_bgr); plt.title('cv2 mentah di plt (kanal tertukar)'); plt.show()

### 5. Setelah citra ditampilkan dengan `figsize` yang jauh lebih besar, apakah nilai `img.shape` ikut berubah? Jelaskan perbedaan antara mengubah ukuran citra dan mengubah ukuran tampilan.

Tidak. `figsize` hanya mengatur ukuran **kanvas** gambar saat ditampilkan (satuan inci);
jumlah piksel array tetap sama sehingga `img.shape` tidak berubah. Hanya `cv.resize()`
yang benar-benar mengubah jumlah piksel (dan karenanya mengubah `img.shape`).

In [ ]:
before = img_rgb.shape
fig = plt.figure(figsize=(14, 14))
ax = fig.add_subplot(1, 1, 1); ax.imshow(img_rgb)
plt.show()
print('shape sebelum figsize:', before)
print('shape sesudah figsize:', img_rgb.shape, '-> tidak berubah')
resized = cv.resize(img_rgb, (347, 229))
print('shape sesudah cv.resize:', resized.shape, '-> berubah')

---
# Tugas Praktikum — Penerapan

### 1. Tampilkan citra hanya pada kombinasi kanal Merah–Biru dan Hijau–Biru, kanal yang tidak dipakai dinolkan. Susun berdampingan dalam satu figure berjudul.

Kita bekerja pada array **RGB** agar penamaan kanal jelas: R = indeks 0, G = indeks 1, B = indeks 2.
Kanal yang tidak dipakai di-nol-kan.

In [ ]:
rgb = cv.cvtColor(cv.imread(PATH), cv.COLOR_BGR2RGB)
rb = rgb.copy(); rb[:, :, 1] = 0     # nolkan Hijau -> sisa Merah+Biru
gb = rgb.copy(); gb[:, :, 0] = 0     # nolkan Merah -> sisa Hijau+Biru

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(rb); ax[0].set_title('Kanal Merah + Biru (Hijau dinolkan)')
ax[1].imshow(gb); ax[1].set_title('Kanal Hijau + Biru (Merah dinolkan)')
fig.suptitle('Kombinasi Kanal Warna')
plt.show()

### 2. Tampilkan citra dalam tiga bentuk pencerminan: vertikal, horizontal, dan keduanya, dalam satu figure berisi tiga subplot berlabel.

`cv.flip`: `0` = vertikal, `1` = horizontal, `-1` = keduanya.

In [ ]:
flip_v = cv.flip(rgb, 0)
flip_h = cv.flip(rgb, 1)
flip_b = cv.flip(rgb, -1)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(flip_v); ax[0].set_title('Flip Vertikal (0)')
ax[1].imshow(flip_h); ax[1].set_title('Flip Horizontal (1)')
ax[2].imshow(flip_b); ax[2].set_title('Flip Vertikal + Horizontal (-1)')
plt.show()

### 3. Gambarkan sebuah `rectangle` dan sebuah `circle` pada bagian wajah/badan foto aktivitas Anda, lalu tambahkan teks berisi nama Anda.

Ubah `PHOTO` ke berkas foto aktivitas Anda. Koordinat dibuat proporsional terhadap
ukuran citra agar tidak bergantung pada resolusi tertentu.

In [ ]:
PHOTO = "/content/KegiatanPCVK.jpg"
foto = cv.cvtColor(cv.imread(PHOTO), cv.COLOR_BGR2RGB)
h, w = foto.shape[:2]
annot = foto.copy()

# Rectangle
cv.rectangle(annot, (10, 1000), (950, 1600),
             color=(255, 0, 0), thickness=3)

# Circle
cv.circle(annot, (550, 800), radius=200,
          color=(0, 255, 0), thickness=4)

# teks nama
cv.putText(annot, 'Singgih Wahyu Permana', (int(0.05 * w), int(0.98 * h)), cv.FONT_HERSHEY_SIMPLEX,
           fontScale=max(0.5, h / 600), color=(255, 255, 0),
           thickness=max(1, h // 300), lineType=cv.LINE_AA)

plt.figure(figsize=(8, 8))
plt.imshow(annot)
plt.title('Anotasi foto aktivitas')
plt.show()

---
# Tugas Praktikum — Analisa

**Pertanyaan:** Berapa persen piksel pada foto KTM yang lebih gelap dari nilai 100?
Apakah mengecilkan lalu membesarkan citra mengubah rata-rata intensitasnya?

**Rancangan program:**
1. Baca citra sebagai grayscale.
2. Hitung persentase piksel dengan nilai < 100.
3. Kecilkan dengan `cv.resize` ke setengah, lalu besarkan kembali ke ukuran semula.
4. Bandingkan rata-rata intensitas citra asli vs hasil downscale–upscale.

**Data & kesimpulan** dicetak oleh sel berikut.

In [ ]:
gray = cv.imread(PATH, cv.IMREAD_GRAYSCALE)
h, w = gray.shape[:2]

dark_ratio = (gray < 100).mean() * 100
print(f'Piksel dengan intensitas < 100: {dark_ratio:.2f}%')

down = cv.resize(gray, (w // 2, h // 2), interpolation=cv.INTER_AREA)
up = cv.resize(down, (w, h), interpolation=cv.INTER_LINEAR)

print(f'Rata-rata intensitas asli            : {gray.mean():.3f}')
print(f'Rata-rata setelah downscale-upscale  : {up.mean():.3f}')
print(f'Selisih                              : {abs(gray.mean() - up.mean()):.3f}')

# Kesimpulan: mengecilkan lalu membesarkan citra cenderung sedikit menurunkan
# rata-rata intensitas karena interpolasi menghaluskan (blur) dan meredam nilai ekstrem.
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(gray, cmap='gray'); plt.title('Asli')
plt.subplot(1, 2, 2); plt.imshow(up, cmap='gray'); plt.title('Downscale lalu upscale')
plt.show()

---
# Tugas Praktikum — Aplikasi

Aturan: hanya memakai fungsi dari modul ini (pembacaan berkas, pengirisan,
`cv2.resize`, `cv2.flip`, `cv2.cvtColor`, fungsi penggambaran OpenCV, kanvas NumPy).
Semua ukuran, posisi, dan ketebalan harus **dihitung dari ukuran citra masukan**,
bukan angka tetap.

## Studi Kasus 1 — Label identitas proporsional

Panitia perlu membubuhkan label pada ratusan foto dokumentasi yang ukurannya
bermacam-macam. Tinggi bar dan ukuran huruf harus menyesuaikan ukuran citra
sehingga proporsinya tetap sama.

Pendekatan: tinggi bar = `tinggi // 12`, posisi teks dan ketebalan dihitung
dari tinggi bar tersebut. Fungsi yang sama dipakai untuk dua citra berbeda ukuran.

In [ ]:
def tambah_label(img_bgr, teks):
    """Tempelkan bar label proporsional di bagian bawah citra.

    Semua ukuran dihitung dari ukuran citra masukan, jadi fungsi yang sama
    tetap seimbang untuk citra berukuran apa pun.
    """
    img = img_bgr.copy()
    h, w = img.shape[:2]
    bar_h = max(12, h // 12)                 # tinggi bar proporsional
    thickness = max(1, bar_h // 15)

    # Ukuran font disesuaikan dengan tinggi bar
    font = cv.FONT_HERSHEY_SIMPLEX
    font_scale = bar_h / 50.0

    # Ukuran teks untuk menentukan posisi vertikal
    (text_w, text_h), baseline = cv.getTextSize(
        teks, font, font_scale, thickness
    )

    cv.rectangle(img, (0, h - bar_h), (w, h),
                 color=(0, 0, 0), thickness=-1)

    cv.putText(img, teks,
               (max(4, w // 100), h - (bar_h - text_h) // 2 - baseline),
               font, font_scale, (255, 255, 255),
               thickness, lineType=cv.LINE_AA)

    return cv.cvtColor(img, cv.COLOR_BGR2RGB)

PATH = "/content/pemandangan.jpg"
img_lanskap = cv.resize(cv.imread(PATH), (900, 600))   # contoh lanskap 900x600
img_potret = cv.resize(cv.imread(PATH), (480, 760))    # contoh potret 480x760

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(tambah_label(img_lanskap, 'Dokumentasi PCVK - 2026'))
ax[0].set_title('Masukan 900 x 600 px')
ax[1].imshow(tambah_label(img_potret, 'Dokumentasi PCVK - 2026'))
ax[1].set_title('Masukan 480 x 760 px')
plt.show()

## Studi Kasus 2 — Thumbnail persegi 1:1 tanpa mengubah rasio

Marketplace menampilkan foto produk dalam bingkai persegi 1:1. Foto potret/lanskap
tidak boleh terpotong atau terlihat tidak proporsional.

Pendekatan: buat kanvas persegi (warna latar), skala citra agar sisi terpanjang
pas di dalam kanvas (rasio asli terjaga), lalu tempelkan **terpusat** dan tambahkan
logo/nama toko berukuran proporsional.

In [ ]:
def thumbnail_persegi(img_bgr, teks_toko, warna_latar=(255, 255, 255)):
    """Ubah citra apa pun menjadi thumbnail persegi 1:1 tanpa mengubah rasio asli.

    Citra diskalakan agar sisi terpanjangnya pas di dalam kanvas persegi,
    lalu diletakkan tepat di tengah dan diberi label nama toko proporsional.
    """
    h, w = img_bgr.shape[:2]
    sisi = max(h, w)                          # sisi kanvas persegi
    skala = sisi / max(h, w)                  # jaga rasio asli
    new_w, new_h = int(w * skala), int(h * skala)
    if (new_w, new_h) != (w, h):
        img_bgr = cv.resize(img_bgr, (new_w, new_h), interpolation=cv.INTER_AREA)

    kanvas = np.full((sisi, sisi, 3), warna_latar, dtype=np.uint8)
    y0 = (sisi - new_h) // 2                   # posisi terpusat
    x0 = (sisi - new_w) // 2
    kanvas[y0:y0 + new_h, x0:x0 + new_w] = img_bgr

    font_scale = sisi / 600.0
    thickness = max(1, sisi // 300)
    cv.putText(kanvas, teks_toko, (max(4, sisi // 40), sisi - max(6, sisi // 40)),
               cv.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255),
               thickness * 3, lineType=cv.LINE_AA)
    cv.putText(kanvas, teks_toko, (max(4, sisi // 40), sisi - max(6, sisi // 40)),
               cv.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0),
               thickness, lineType=cv.LINE_AA)
    return cv.cvtColor(kanvas, cv.COLOR_BGR2RGB)


produk_potret = cv.resize(cv.imread(PATH), (480, 760))
produk_lanskap = cv.resize(cv.imread(PATH), (900, 600))

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(thumbnail_persegi(produk_potret, 'Toko Saya'))
ax[0].set_title('Dari 480 x 760 (potret)')
ax[1].imshow(thumbnail_persegi(produk_lanskap, 'Toko Saya'))
ax[1].set_title('Dari 900 x 600 (lanskap)')
plt.show()

---
### Selesai

Simpan notebook ini sebagai `Week02.ipynb` via *File → Save a copy in GitHub*.